# 🎬 AI Video Generator - Google Colab

**Resimden video oluştur, müzik ekle, yüz taşı - Tamamen Ücretsiz!**

---

## ⚠️ ÖNEMLİ
1. **Runtime tipi değiştir**: Menu > Runtime > Change runtime type > GPU (T4 veya P100)
2. Bu hücreler sırasıyla çalıştırılmalıdır (yukarıdan aşağıya)
3. Her hücreden sonra çıktıyı kontrol edin

## 1️⃣ GPU Kontrol Et & Kütüphaneleri Kur

In [ ]:
# GPU Kontrol
import subprocess
import sys

print("🔍 GPU Kontrolü...")
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

import torch
print(f"\n✅ CUDA Kullanılabilir: {torch.cuda.is_available()}")
print(f"✅ GPU Adı: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Yok'}")
print(f"✅ CUDA Versiyon: {torch.version.cuda}")

## 2️⃣ Kütüphaneleri Kur (5-10 dakika alabilir)

In [ ]:
# Temel kütüphaneler
print("📦 Kütüphaneler kuruluyor...\n")

!pip install -q opencv-python
!pip install -q librosa
!pip install -q pillow
!pip install -q scipy
!pip install -q ffmpeg-python
!pip install -q imageio
!pip install -q imageio-ffmpeg
!pip install -q transformers
!pip install -q safetensors
!pip install -q omegaconf
!pip install -q einops
!pip install -q deep-translator

print("\n✅ Temel kütüphaneler kuruldu!")

## 3️⃣ PyTorch ve Diffusers Kur

In [ ]:
print("🤖 PyTorch ve Diffusers kuruluyor...\n")

!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118
!pip install -q diffusers

print("\n✅ PyTorch ve Diffusers kuruldu!")

## 4️⃣ Video Generator Kodunu Tanımla

In [ ]:
import torch
import cv2
import numpy as np
from PIL import Image
import librosa
from pathlib import Path
import os
from diffusers import DiffusionPipeline, DDIMScheduler
import imageio
from scipy.interpolate import interp1d
import subprocess
import warnings
warnings.filterwarnings('ignore')

class ImageToVideoGenerator:
    """Resimden video oluşturucu"""
    
    def __init__(self, device='cuda' if torch.cuda.is_available() else 'cpu'):
        self.device = device
        print(f"Device: {self.device}")
        self.pipe = None
        
    def load_model(self):
        """Zeroscope modelini yükle"""
        print("🤖 Model yükleniyor (ilk kez 5-10 dakika alabilir)...")
        
        scheduler = DDIMScheduler.from_pretrained(
            "cerspense/zeroscope_v2_576w",
            subfolder="scheduler"
        )
        
        self.pipe = DiffusionPipeline.from_pretrained(
            "cerspense/zeroscope_v2_576w",
            scheduler=scheduler,
            torch_dtype=torch.float16 if self.device == 'cuda' else torch.float32
        )
        self.pipe = self.pipe.to(self.device)
        self.pipe.enable_attention_slicing()
        
        print("✅ Model yüklendi!")
        
    def generate_video_from_image(self, image_path, prompt, duration_seconds=6, fps=24, num_frames=24):
        """Resimden video oluştur"""
        
        if self.pipe is None:
            self.load_model()
        
        # Resmi yükle ve işle
        image = Image.open(image_path).convert("RGB")
        image = image.resize((576, 576))
        
        print(f"🎬 Video oluşturuluyor: {prompt}")
        print(f"   Süre: {duration_seconds}s, FPS: {fps}, Frames: {num_frames}")
        
        # Video oluştur
        with torch.no_grad():
            video_frames = self.pipe(
                prompt=prompt,
                image=image,
                num_inference_steps=40,
                height=576,
                width=576,
                num_frames=num_frames,
                guidance_scale=7.5
            ).frames
        
        return video_frames
    
    def save_video(self, frames, output_path, fps=24):
        """Video dosyasını kaydet"""
        print(f"💾 Video kaydediliyor: {output_path}")
        
        writer = imageio.get_writer(output_path, fps=fps)
        for frame in frames:
            frame_array = np.array(frame)
            writer.append_data(frame_array)
        writer.close()
        
        print(f"✅ Video kaydedildi!")

class AudioProcessor:
    """Ses işleme"""
    
    @staticmethod
    def get_audio_duration(audio_path):
        """Ses dosyasının uzunluğunu al"""
        y, sr = librosa.load(audio_path)
        duration = librosa.get_duration(y=y, sr=sr)
        return duration
    
    @staticmethod
    def add_audio_to_video(video_path, audio_path, output_path):
        """Videoya ses ekle"""
        print(f"🔊 Ses videoya ekleniyor...")
        
        cmd = [
            'ffmpeg',
            '-i', video_path,
            '-i', audio_path,
            '-c:v', 'copy',
            '-c:a', 'aac',
            '-map', '0:v:0',
            '-map', '1:a:0',
            '-shortest',
            '-y',
            output_path
        ]
        
        subprocess.run(cmd, capture_output=True)
        print("✅ Ses eklendi!")

print("✅ Kodlar tanımlandı!")

## 5️⃣ Resim Yükle

In [ ]:
from google.colab import files
import os

# Klasörleri oluştur
os.makedirs('input', exist_ok=True)
os.makedirs('output', exist_ok=True)

print("📸 Lütfen resminizi seçin...\n")
uploaded_images = files.upload()

# Resmi input klasörüne kopyala
image_path = None
for filename in uploaded_images:
    if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
        image_path = f'input/{filename}'
        os.rename(filename, image_path)
        print(f"✅ Resim kaydedildi: {image_path}")
        break

if not image_path:
    print("❌ Geçerli resim bulunamadı!")

## 6️⃣ Müzik Yükle

In [ ]:
from google.colab import files

print("🎵 Lütfen müziğinizi seçin (MP3 veya WAV)...\n")
uploaded_audio = files.upload()

# Müziği input klasörüne kopyala
audio_path = None
for filename in uploaded_audio:
    if filename.lower().endswith(('.mp3', '.wav', '.m4a')):
        audio_path = f'input/{filename}'
        os.rename(filename, audio_path)
        print(f"✅ Müzik kaydedildi: {audio_path}")
        
        # Ses uzunluğunu kontrol et
        duration = AudioProcessor.get_audio_duration(audio_path)
        print(f"🔊 Ses süresi: {duration:.2f} saniye")
        
        if duration > 360:  # 6 dakika
            print("⚠️  Ses 6 dakikadan uzun (360 saniye). Model en fazla 6 saniye video oluşturur.")
        break

if not audio_path:
    print("❌ Geçerli müzik bulunamadı!")

## 7️⃣ Prompt Gir (Türkçe veya İngilizce)

In [ ]:
from deep_translator import GoogleTranslator

# Prompt gir
prompt_input = input("\n📝 Videonun açıklamasını girin (Türkçe veya İngilizce):\n> ")

# Türkçe ise çevir
if any(ord(c) > 127 for c in prompt_input):  # Türkçe karakterler var mı?
    print("\n🌐 Türkçe prompt algılandı, çeviriliyor...")
    try:
        translator = GoogleTranslator(source_language='tr', target_language='en')
        prompt = translator.translate(prompt_input)
        print(f"✅ Çevirisi: {prompt}")
    except:
        prompt = prompt_input
        print(f"⚠️  Çeviri başarısız, orijinal kullanılıyor: {prompt}")
else:
    prompt = prompt_input
    print(f"✅ Prompt: {prompt}")

print(f"\n🎬 Video oluşturma için prompt: {prompt}")

## 8️⃣ Video Oluştur (20-30 dakika alabilir)

In [ ]:
print("\n" + "="*60)
print("🎬 VIDEO OLUŞTURMA BAŞLIYOR")
print("="*60)
print("⏱️  Bu işlem 20-30 dakika alabilir, lütfen bekleyin...\n")

# Generator oluştur
generator = ImageToVideoGenerator()

# Video oluştur
frames = generator.generate_video_from_image(
    image_path=image_path,
    prompt=prompt,
    duration_seconds=6,
    fps=24,
    num_frames=24
)

# Video kaydet
temp_video = 'output/video_temp.mp4'
generator.save_video(frames, temp_video, fps=24)

print("\n✅ Video oluşturuldu!")

## 9️⃣ Ses Ekle

In [ ]:
print("\n" + "="*60)
print("🔊 SES EKLENİYOR")
print("="*60 + "\n")

final_video = 'output/final_video.mp4'

AudioProcessor.add_audio_to_video(
    video_path=temp_video,
    audio_path=audio_path,
    output_path=final_video
)

print("\n✅ Ses eklendi!")

## 🔟 Videoyu Göster & İndir

In [ ]:
from IPython.display import Video
import os

print("\n" + "="*60)
print("✨ TAMAMLANDI!")
print("="*60)

print(f"\n📁 Final Video: {final_video}")
print(f"📊 Dosya Boyutu: {os.path.getsize(final_video) / (1024*1024):.2f} MB")

print("\n📹 Video Ön İzlemesi:\n")
Video(final_video)

## 1️⃣1️⃣ Videoyu İndir

In [ ]:
from google.colab import files

print("💾 Video indirilmeye hazırlanıyor...\n")

files.download(final_video)

print("✅ İndirme başladı!")
print("   Video tarayıcının İndirilenler klasörüne kaydedildi.")

---

## 📝 Notlar

✅ **Başarı!** Videonuz oluşturuldu ve indirilmeye hazır.

💡 **İpuçları:**
- Prompt'u detaylı yazarsanız daha iyi sonuç alırsınız
- Türkçe prompt yazabilirsiniz, otomatik çevrilir
- Farklı promptlar deneyin

⚠️ **Hatalar?**
- "CUDA out of memory" → Colab'ı yeniden başlatın
- "Model yüklenmedi" → İnternet bağlantısını kontrol edin
- İşlem durdu → Runtime yenile (Runtime > Restart runtime)

---

**Keyifli video oluşturmalar! 🎥✨**